# Stage 02 — Residual-attention corrector for periodic advection

Python/Jupyter conversion of `STAGE_02_INF_RBF_ADV_PAPER.m`. Adaptivity changes only the PDE collocation density; the periodic infinite-width Gaussian-RBF kernel is unchanged.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eigh

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import erf
from pathlib import Path
from time import perf_counter

def radical_inverse(n, b):
    n = np.asarray(n, dtype=int).copy()
    r = np.zeros_like(n, dtype=float)
    f = 1.0 / b
    while np.any(n > 0):
        r += f * (n % b)
        n //= b
        f /= b
    return r

def halton_2d_prefix(N):
    idx = np.arange(1, N+1)
    return radical_inverse(idx, 2), radical_inverse(idx, 3)

def dilate_periodic(mask, radius):
    mask = np.asarray(mask, dtype=bool)
    N = mask.size
    out = np.zeros_like(mask)
    for j in np.flatnonzero(mask):
        for r in range(-radius, radius+1):
            out[(j+r) % N] = True
    return out

def attention_to_levels(score, theta_mid, theta_high, high_radius, mid_radius):
    score = np.asarray(score, dtype=float).ravel()
    if np.max(score) <= np.finfo(float).eps:
        return np.ones(score.size, dtype=int)
    high_raw = score >= theta_high
    mid_raw = score >= theta_mid
    if not np.any(high_raw):
        high_raw[np.argmax(score)] = True
    high_mask = dilate_periodic(high_raw, high_radius)
    mid_mask = dilate_periodic(mid_raw | high_raw, mid_radius)
    levels = np.ones(score.size, dtype=int)
    levels[mid_mask] = 2
    levels[high_mask] = 3
    return levels

def rbf1d_uniform_derivs(x, y, sigma, a, b):
    x = np.asarray(x, dtype=float).reshape(-1, 1)
    y = np.asarray(y, dtype=float).reshape(1, -1)
    m = 0.5*(x+y)
    d = x-y
    za = a-m
    zb = b-m
    Ed = np.exp(-(d**2)/(4*sigma**2))
    ea = np.exp(-(za/sigma)**2)
    eb = np.exp(-(zb/sigma)**2)
    E = erf(zb/sigma)-erf(za/sigma)
    J0 = 0.5*sigma*np.sqrt(np.pi)*E
    J1 = 0.5*sigma**2*(ea-eb)
    J2 = 0.25*sigma**3*np.sqrt(np.pi)*E - 0.5*sigma**2*(zb*eb-za*ea)
    C = 1/(b-a)
    K = C*Ed*J0
    Ix = 0.5*d*J0 - J1
    Iy = -0.5*d*J0 - J1
    K1 = -C*Ed*Ix/sigma**2
    K2 = -C*Ed*Iy/sigma**2
    K12 = C*Ed*(J2 - 0.25*d**2*J0)/sigma**4
    return K, K1, K2, K12

def compute_kernel_norm(cfg):
    xr = 0.5*(cfg["XL"]+cfg["XR"])
    tr = 0.5*cfg["dt"]
    Kx = rbf1d_uniform_derivs([xr],[xr],cfg["sigma_x"],cfg["cx_a"],cfg["cx_b"])[0]
    Kt = rbf1d_uniform_derivs([tr],[tr],cfg["sigma_t"],cfg["ct_a"],cfg["ct_b"])[0]
    return float((Kx*Kt)[0,0])

def kernel_2d(cfg, x1,t1,x2,t2,type1,type2):
    Kx,Kx1,Kx2,Kx12 = rbf1d_uniform_derivs(x1,x2,cfg["sigma_x"],cfg["cx_a"],cfg["cx_b"])
    Kt,Kt1,Kt2,Kt12 = rbf1d_uniform_derivs(t1,t2,cfg["sigma_t"],cfg["ct_a"],cfg["ct_b"])
    if type1=="val" and type2=="val":
        K = Kx*Kt
    elif type1=="pde" and type2=="val":
        K = Kx1*Kt + Kx*Kt1
    elif type1=="val" and type2=="pde":
        K = Kx2*Kt + Kx*Kt2
    elif type1=="pde" and type2=="pde":
        K = Kx12*Kt + Kx1*Kt2 + Kx2*Kt1 + Kx*Kt12
    else:
        raise ValueError("Unknown kernel functional type.")
    return K/cfg["norm_const"]

def kernel_val_vs_bc(cfg,x,t,tB):
    tB=np.asarray(tB)
    return (kernel_2d(cfg,x,t,np.full_like(tB,cfg["XL"],dtype=float),tB,"val","val")
            - kernel_2d(cfg,x,t,np.full_like(tB,cfg["XR"],dtype=float),tB,"val","val"))

def kernel_pde_vs_bc(cfg,x,t,tB):
    tB=np.asarray(tB)
    return (kernel_2d(cfg,x,t,np.full_like(tB,cfg["XL"],dtype=float),tB,"pde","val")
            - kernel_2d(cfg,x,t,np.full_like(tB,cfg["XR"],dtype=float),tB,"pde","val"))

def kernel_bc_vs_val(cfg,tB,x,t):
    tB=np.asarray(tB)
    return (kernel_2d(cfg,np.full_like(tB,cfg["XL"],dtype=float),tB,x,t,"val","val")
            - kernel_2d(cfg,np.full_like(tB,cfg["XR"],dtype=float),tB,x,t,"val","val"))

def kernel_bc_vs_bc(cfg,t1,t2):
    t1=np.asarray(t1); t2=np.asarray(t2)
    XL1=np.full_like(t1,cfg["XL"],dtype=float); XR1=np.full_like(t1,cfg["XR"],dtype=float)
    XL2=np.full_like(t2,cfg["XL"],dtype=float); XR2=np.full_like(t2,cfg["XR"],dtype=float)
    return (kernel_2d(cfg,XL1,t1,XL2,t2,"val","val")
            - kernel_2d(cfg,XL1,t1,XR2,t2,"val","val")
            - kernel_2d(cfg,XR1,t1,XL2,t2,"val","val")
            + kernel_2d(cfg,XR1,t1,XR2,t2,"val","val"))

def build_constraint_gram(cfg,tr):
    xP,tP,tB,xI,tI = tr["xP"],tr["tP"],tr["tB"],tr["xI"],tr["tI"]
    wP,wB,wI = tr["wP"],tr["wB"],tr["wI"]
    KPP=kernel_2d(cfg,xP,tP,xP,tP,"pde","pde")
    KPB=kernel_pde_vs_bc(cfg,xP,tP,tB)
    KPI=kernel_2d(cfg,xP,tP,xI,tI,"pde","val")
    KBB=kernel_bc_vs_bc(cfg,tB,tB)
    KBI=kernel_bc_vs_val(cfg,tB,xI,tI)
    KII=kernel_2d(cfg,xI,tI,xI,tI,"val","val")
    K=np.block([
        [wP*wP*KPP, wP*wB*KPB, wP*wI*KPI],
        [wB*wP*KPB.T, wB*wB*KBB, wB*wI*KBI],
        [wI*wP*KPI.T, wI*wB*KBI.T, wI*wI*KII],
    ])
    return 0.5*(K+K.T)

def predict_value(cfg,tr,a,xq,tq):
    NP=len(tr["xP"]); NB=len(tr["tB"])
    aP=a[:NP]; aB=a[NP:NP+NB]; aI=a[NP+NB:]
    KP=kernel_2d(cfg,xq,tq,tr["xP"],tr["tP"],"val","pde")
    KB=kernel_val_vs_bc(cfg,xq,tq,tr["tB"])
    KI=kernel_2d(cfg,xq,tq,tr["xI"],tr["tI"],"val","val")
    return (tr["wP"]*KP)@aP + (tr["wB"]*KB)@aB + (tr["wI"]*KI)@aI

def predict_pde_residual(cfg,tr,a,xq,tq):
    NP=len(tr["xP"]); NB=len(tr["tB"])
    aP=a[:NP]; aB=a[NP:NP+NB]; aI=a[NP+NB:]
    KP=kernel_2d(cfg,xq,tq,tr["xP"],tr["tP"],"pde","pde")
    KB=kernel_pde_vs_bc(cfg,xq,tq,tr["tB"])
    KI=kernel_2d(cfg,xq,tq,tr["xI"],tr["tI"],"pde","val")
    return (tr["wP"]*KP)@aP + (tr["wB"]*KB)@aB + (tr["wI"]*KI)@aI

In [3]:
P=np.load("STAGE_01_INF_RBF_ADV_attention_PAPER.npz",allow_pickle=False)

XL,XR,L=map(float,(P["XL"],P["XR"],P["L"]))
pulse_beta=float(P["pulse_beta"])
NBx,nSlabs=int(P["NBx"]),int(P["nSlabs"])
dt_slab,Tfinal=float(P["dt_slab"]),float(P["Tfinal"])
x_edges=P["x_edges"]
Nic,Nbc=int(P["Nic"]),int(P["Nbc"])
xI,tB=P["xI"],P["tB"]
sigma_x,sigma_t=float(P["sigma_x"]),float(P["sigma_t"])
cx_a,cx_b,ct_a,ct_b=map(float,(P["cx_a"],P["cx_b"],P["ct_a"],P["ct_b"]))
ridge_rel,eig_tol_rel=float(P["ridge_rel"]),float(P["eig_tol_rel"])
Lmat_stage1=P["Lmat"].astype(int)

def init(x):
    x=np.asarray(x,float)
    return np.exp(-(pulse_beta*x)**2)

def uexact(x,t):
    x=np.asarray(x,float)
    return init(np.mod(x-t-XL,L)+XL)

Npde_level=np.array([8,16,24],dtype=int)
Nmax_block=int(Npde_level[-1])

kcfg=dict(sigma_x=sigma_x,sigma_t=sigma_t,cx_a=cx_a,cx_b=cx_b,
          ct_a=ct_a,ct_b=ct_b,XL=XL,XR=XR,dt=dt_slab)
kcfg["norm_const"]=compute_kernel_norm(kcfg)

In [4]:
_t_stage2_total0 = perf_counter()

hx01,ht01=halton_2d_prefix(Nmax_block)
NPmax=NBx*Nmax_block
xPmax=np.zeros(NPmax); tPmax=np.zeros(NPmax)
block_id=np.zeros(NPmax,dtype=int); local_id=np.zeros(NPmax,dtype=int)
kk=0
for jx in range(NBx):
    xa,xb=x_edges[jx],x_edges[jx+1]
    sl=slice(kk,kk+Nmax_block)
    xPmax[sl]=xa+(xb-xa)*hx01
    tPmax[sl]=dt_slab*ht01
    block_id[sl]=jx
    local_id[sl]=np.arange(1,Nmax_block+1)
    kk+=Nmax_block

tI=np.zeros_like(xI)
full=dict(xP=xPmax,tP=tPmax,tB=tB,xI=xI,tI=tI,wP=1.0,wB=1.0,wI=1.0)

_t0 = perf_counter()
Kfull_unweighted=build_constraint_gram(kcfg,full)
t_stage2_full_assembly = perf_counter() - _t0

idxB_full=NPmax+np.arange(Nbc)
idxI_full=NPmax+Nbc+np.arange(Nic)

print(f"Full operator-kernel assembly = {t_stage2_full_assembly:.6f} s")


Full operator-kernel assembly = 0.102643 s


In [5]:
uI=init(xI)
Nx_plot=801
x_plot=np.linspace(XL,XR,Nx_plot)
plot_times=np.arange(nSlabs+1)*dt_slab
Uplot=np.zeros((nSlabs+1,Nx_plot))
Uplot[0]=init(x_plot)
Npde_used=np.zeros(nSlabs,dtype=int)
rank_used=np.zeros(nSlabs,dtype=int)
level_used=np.zeros((nSlabs,NBx),dtype=int)
cache={}

t_stage2_pattern_assembly = 0.0
t_stage2_factorization = 0.0
t_stage2_coeff = 0.0
t_stage2_prediction = 0.0
cache_hits = 0
cache_misses = 0

for s in range(nSlabs):
    lev=Lmat_stage1[s].astype(int)
    level_used[s]=lev

    active=np.zeros(NPmax,dtype=bool)
    for jx in range(NBx):
        nloc=Npde_level[lev[jx]-1]
        active |= (block_id==jx)&(local_id<=nloc)
    idxP=np.flatnonzero(active)
    Np=len(idxP)
    Npde_used[s]=Np

    wP,wB,wI=1/np.sqrt(Np),1/np.sqrt(Nbc),1/np.sqrt(Nic)
    idx=np.r_[idxP,idxB_full,idxI_full]
    scale=np.r_[np.full(Np,wP),np.full(Nbc,wB),np.full(Nic,wI)]

    key=tuple(lev.tolist())
    if key in cache:
        Vuse,duse,rank=cache[key]
        cache_hits += 1
    else:
        cache_misses += 1

        _t0 = perf_counter()
        Ksub=Kfull_unweighted[np.ix_(idx,idx)]
        Ksub=(scale[:,None]*scale[None,:])*Ksub
        Ksub=0.5*(Ksub+Ksub.T)
        t_stage2_pattern_assembly += perf_counter() - _t0

        _t0 = perf_counter()
        d,V=eigh(Ksub)
        dmax=np.max(d)
        if dmax<=0:
            raise RuntimeError(f"Corrector Gram matrix is not positive on slab {s+1}.")
        keep=d>eig_tol_rel*dmax
        lam=ridge_rel*dmax
        Vuse=V[:,keep]; duse=d[keep]+lam; rank=int(np.sum(keep))
        t_stage2_factorization += perf_counter() - _t0

        cache[key]=(Vuse,duse,rank)
    rank_used[s]=rank

    rhs=np.r_[np.zeros(Np),np.zeros(Nbc),wI*uI]
    _t0 = perf_counter()
    coeff=Vuse@((Vuse.T@rhs)/duse)
    t_stage2_coeff += perf_counter() - _t0

    tr=dict(xP=xPmax[idxP],tP=tPmax[idxP],tB=tB,xI=xI,tI=tI,wP=wP,wB=wB,wI=wI)

    _t0 = perf_counter()
    uq_top=predict_value(kcfg,tr,coeff,x_plot,np.full_like(x_plot,dt_slab))
    Uplot[s+1]=uq_top
    uI=predict_value(kcfg,tr,coeff,xI,np.full_like(xI,dt_slab))
    t_stage2_prediction += perf_counter() - _t0

u_ex_final=uexact(x_plot,Tfinal)
err_final=Uplot[-1]-u_ex_final
corrector_rmse=np.sqrt(np.mean(err_final**2))
corrector_max_error=np.max(np.abs(err_final))
avg_Npde=float(np.mean(Npde_used))
min_Npde=int(np.min(Npde_used))
max_Npde=int(np.max(Npde_used))
unique_level_patterns=len(cache)

t_stage2_core = (
    t_stage2_full_assembly
    + t_stage2_pattern_assembly
    + t_stage2_factorization
    + t_stage2_coeff
)
t_stage2_total = perf_counter() - _t_stage2_total0

print(f"Corrector RMSE={corrector_rmse:.3e}")
print(f"Corrector max error={corrector_max_error:.3e}")
print(f"Avg/min/max PDE points={avg_Npde:.2f}/{min_Npde}/{max_Npde}")
print(f"Unique level patterns={unique_level_patterns}, cache hits/misses={cache_hits}/{cache_misses}")
print(f"Pattern matrix extraction/scale = {t_stage2_pattern_assembly:.6f} s")
print(f"Cached spectral factorizations  = {t_stage2_factorization:.6f} s")
print(f"Coefficient solves (all slabs)  = {t_stage2_coeff:.6f} s")
print(f"Slab prediction/transfer        = {t_stage2_prediction:.6f} s")
print(f"Corrector core CFKS time        = {t_stage2_core:.6f} s")
print(f"Stage-2 total elapsed time      = {t_stage2_total:.6f} s")


Corrector RMSE=2.871e-03
Corrector max error=2.128e-02
Avg/min/max PDE points=307.04/288/320
Unique level patterns=52, cache hits/misses=48/52
Pattern matrix extraction/scale = 0.183284 s
Cached spectral factorizations  = 4.297981 s
Coefficient solves (all slabs)  = 0.054114 s
Slab prediction/transfer        = 15.487125 s
Corrector core CFKS time        = 4.638022 s
Stage-2 total elapsed time      = 20.156522 s


In [6]:
np.savez_compressed(
    "STAGE_02_INF_RBF_ADV_results_PAPER.npz",
    XL=XL,XR=XR,L=L,pulse_beta=pulse_beta,NBx=NBx,x_edges=x_edges,
    nSlabs=nSlabs,dt_slab=dt_slab,Tfinal=Tfinal,Nic=Nic,Nbc=Nbc,xI=xI,tB=tB,
    Npde_level=Npde_level,Nmax_block=Nmax_block,
    sigma_x=sigma_x,sigma_t=sigma_t,cx_a=cx_a,cx_b=cx_b,ct_a=ct_a,ct_b=ct_b,
    ridge_rel=ridge_rel,eig_tol_rel=eig_tol_rel,
    xPmax=xPmax,tPmax=tPmax,block_id=block_id,local_id=local_id,
    x_plot=x_plot,plot_times=plot_times,Uplot=Uplot,Npde_used=Npde_used,rank_used=rank_used,
    level_used=level_used,corrector_rmse=corrector_rmse,corrector_max_error=corrector_max_error,
    avg_Npde=avg_Npde,min_Npde=min_Npde,max_Npde=max_Npde,
    unique_level_patterns=unique_level_patterns,u_ex_final=u_ex_final,err_final=err_final,
    t_stage2_full_assembly=t_stage2_full_assembly,
    t_stage2_pattern_assembly=t_stage2_pattern_assembly,
    t_stage2_factorization=t_stage2_factorization,
    t_stage2_coeff=t_stage2_coeff,
    t_stage2_prediction=t_stage2_prediction,
    t_stage2_core=t_stage2_core,
    t_stage2_total=t_stage2_total,
    cache_hits=cache_hits,cache_misses=cache_misses
)
print("Saved STAGE_02_INF_RBF_ADV_results_PAPER.npz")

Saved STAGE_02_INF_RBF_ADV_results_PAPER.npz
